In [8]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "3581629a",
   "metadata": {},
   "source": [
    "# Customer Lifetime Value (CLV) Prediction Notebook\n",
    "\n",
    "This notebook demonstrates how to build a CLV prediction model using supervised regression models. Unlike an approach that assumes column names, this notebook starts by analyzing the dataset columns and then continues with the following workflow:\n",
    "\n",
    "1. **Data Loading & Analysis:** Read the dataset, display column names, data types, and a preview of the data.\n",
    "2. **Exploratory Data Analysis (EDA):** Generate visualizations and summary statistics to understand the data.\n",
    "3. **Feature Engineering:** Create historical features such as recency, frequency, monetary value, and tenure based on a selected cutoff date. The CLV target (`base_clv`) is calculated using future revenue beyond this cutoff.\n",
    "4. **Model Training & Evaluation:** Train several regression models (Linear Regression, Decision Tree, Random Forest, XGBoost, and LightGBM) and compare their performance using RMSE.\n",
    "5. **Hyperparameter Tuning:** An example on tuning model parameters using GridSearchCV.\n",
    "\n",
    "Feel free to adjust the cutoff date, feature definitions, or modeling choices to suit your needs."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "c7f4e775",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import necessary libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "from datetime import timedelta\n",
    "from sklearn.model_selection import train_test_split, GridSearchCV\n",
    "from sklearn.metrics import mean_squared_error\n",
    "\n",
    "# Regression models\n",
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.tree import DecisionTreeRegressor\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "\n",
    "# XGBoost and LightGBM (if not installed, run: pip install xgboost lightgbm)\n",
    "import xgboost as xgb\n",
    "import lightgbm as lgb\n",
    "\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "388f8949",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the dataset\n",
    "data_path = 'synthetic_customer_transactions.csv'\n",
    "try:\n",
    "    data = pd.read_csv(data_path)\n",
    "    print('Dataset loaded successfully.')\n",
    "except Exception as e:\n",
    "    print(f\"Error loading dataset: {e}\")\n",
    "\n",
    "# Display general information about the dataset\n",
    "print(\"\\nDataset Columns:\")\n",
    "print(data.columns.tolist())\n",
    "\n",
    "print(\"\\nData types:\")\n",
    "print(data.dtypes)\n",
    "\n",
    "print(\"\\nFirst 5 rows of the dataset:\")\n",
    "display(data.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a49ed76c",
   "metadata": {},
   "source": [
    "## Exploratory Analysis of the Dataset\n",
    "\n",
    "Now that we have loaded the data, let’s inspect its shape, summary statistics, and a preview of the data. This helps in verifying which columns are available and will be used for the CLV calculation. \n",
    "\n",
    "For this notebook, it is assumed that the dataset contains transaction-level data. (If the column names are different from what the subsequent code expects, please adjust them accordingly.)\n",
    "\n",
    "*We expect to see columns resembling identifiers for customers, dates of transactions, and revenue amounts.*"
   ]
  },
  {
   "cell_type": "code",
 "execution_count": None,
   "id": "3f2078a8",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary statistics and data overview\n",
    "print(\"\\nDataset Shape:\", data.shape)\n",
    "\n",
    "print(\"\\nSummary Statistics:\")\n",
    "display(data.describe(include='all'))\n",
    "\n",
    "print(\"\\nMissing Values by Column:\")\n",
    "print(data.isnull().sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "434fdf43",
   "metadata": {},
   "source": [
    "## Data Preparation and Feature Engineering\n",
    "\n",
    "In order to build a CLV model, we need to create features from historical data and a target variable using future revenue. \n",
    "\n",
    "The following assumptions are made (please adjust if needed based on your actual column names):\n",
    "\n",
    "- **CustomerID:** Unique identifier for each customer.\n",
    "- **TransactionDate:** The date when the transaction occurred. This column is converted to datetime.\n",
    "- **Revenue:** The revenue generated in a given transaction.\n",
    "\n",
    "If your dataset uses different column names, modify the code below accordingly.\n",
    "\n",
    "We will select a cutoff date (here, calculated at the 80th percentile of the transaction date range) to split historical data from future data. Features such as recency (days since last transaction), frequency (number of transactions), monetary (total revenue), and tenure (number of days since the first transaction) will be computed using data before the cutoff. The target variable, `base_clv`, is the sum of revenue for transactions after the cutoff date."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "bba1f1b5",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Ensure the necessary columns are present\n",
    "required_cols = ['CustomerID', 'TransactionDate', 'Revenue']\n",
    "for col in required_cols:\n",
    "    if col not in data.columns:\n",
    "        raise ValueError(f\"Column '{col}' is missing from the dataset. Please adjust the column names accordingly.\")\n",
    "\n",
    "# Convert TransactionDate to datetime\n",
    "data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])\n",
    "\n",
    "# Determine cutoff date based on 80% of the transaction date range\n",
    "min_date = data['TransactionDate'].min()\n",
    "max_date = data['TransactionDate'].max()\n",
    "cutoff_date = min_date + (max_date - min_date) * 0.8\n",
    "print(f\"Cutoff Date: {cutoff_date.date()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "03d118f3",
   "metadata": {},
   "outputs": [],
   "source": [
    "def compute_customer_features(df, cutoff):\n",
    "    \"\"\"Compute historical features and target CLV for each customer. \n",
    "    \n",
    "    Parameters:\n",
    "      - df: DataFrame with transaction data\n",
    "      - cutoff: datetime to separate historical from future data\n",
    "    \n",
    "    Returns:\n",
    "      - DataFrame with index as CustomerID containing historical features and target variable 'base_clv'.\n",
    "    \"\"\"\n",
    "    # Historical and future transactions\n",
    "    historical = df[df['TransactionDate'] <= cutoff]\n",
    "    future = df[df['TransactionDate'] > cutoff]\n",
    "    \n",
    "    # Compute historical features for each customer\n",
    "    historical_features = historical.groupby('CustomerID').agg({\n",
    "        'TransactionDate': [np.min, np.max, 'count'],\n",
    "        'Revenue': 'sum'\n",
    "    })\n",
    "    historical_features.columns = ['FirstTransaction', 'LastTransaction', 'Frequency', 'Monetary']\n",
    "    \n",
    "    # Compute additional features\n",
    "    historical_features['Recency'] = (cutoff - historical_features['LastTransaction']).dt.days\n",
    "    historical_features['Tenure'] = (cutoff - historical_features['FirstTransaction']).dt.days\n",
    "    \n",
    "    # Compute future revenue as the target variable\n",
    "    future_revenue = future.groupby('CustomerID')['Revenue'].sum().to_frame('base_clv')\n",
    "    \n",
    "    # Merge historical features and future revenue\n",
    "    features = historical_features.merge(future_revenue, left_index=True, right_index=True, how='left')\n",
    "    features['base_clv'] = features['base_clv'].fillna(0)\n",
    "    \n",
    "    return features\n",
    "\n",
    "# Generate customer-level feature dataset\n",
    "customer_data = compute_customer_features(data, cutoff_date)\n",
    "print('Customer feature dataset shape:', customer_data.shape)\n",
    "display(customer_data.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3fcab4cf",
   "metadata": {},
   "source": [
    "## Visual Exploratory Analysis of Engineered Features\n",
    "\n",
    "Let's visualize the distributions of the engineered features and the target variable (`base_clv`). We will create histograms and pair plots to understand the data patterns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "59e824f3",
   "metadata": {},
   "outputs": [],
   "source": [
    "# List of features to plot\n",
    "features_to_plot = ['Recency', 'Frequency', 'Monetary', 'Tenure', 'base_clv']\n",
    "\n",
    "plt.figure(figsize=(15, 10))\n",
    "for i, col in enumerate(features_to_plot, 1):\n",
    "    plt.subplot(2, 3, i)\n",
    "    sns.histplot(customer_data[col], kde=True)\n",
    "    plt.title(f\"Distribution of {col}\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Pairplot of features (excluding index)\n",
    "sns.pairplot(customer_data.reset_index()[features_to_plot])\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "73fb4fae",
   "metadata": {},
   "source": [
    "## Preparing Data for Modeling\n",
    "\n",
    "We now define our features (X) and the target variable (y) from the customer-level data. In this example, the features used are the engineered metrics: recency, frequency, monetary, and tenure. \n",
    "\n",
    "After that, we split the data into training and testing sets."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "06a88a36",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select features and target variable\n",
    "X = customer_data[['Recency', 'Frequency', 'Monetary', 'Tenure']]\n",
    "y = customer_data['base_clv']\n",
    "\n",
    "# Split into train and test sets\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n",
    "\n",
    "print('Training set shape:', X_train.shape)\n",
    "print('Test set shape:', X_test.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "90f4ec63",
   "metadata": {},
   "source": [
    "## Model Training and Evaluation\n",
    "\n",
    "We now train several supervised regression models to predict the CLV (`base_clv`). The models included are:\n",
    "\n",
    "- **Linear Regression**\n",
    "- **Decision Tree**\n",
    "- **Random Forest**\n",
    "- **XGBoost**\n",
    "- **LightGBM**\n",
    "\n",
    "The model performances are evaluated using the root mean squared error (RMSE)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "ef3a463c",
   "metadata": {},
   "outputs": [],
   "source": [
    "models = {\n",
    "    'Linear Regression': LinearRegression(),\n",
    "    'Decision Tree': DecisionTreeRegressor(random_state=42),\n",
    "    'Random Forest': RandomForestRegressor(random_state=42),\n",
    "    'XGBoost': xgb.XGBRegressor(random_state=42, objective='reg:squarederror'),\n",
    "    'LightGBM': lgb.LGBMRegressor(random_state=42)\n",
    "}\n",
    "\n",
    "results = {}\n",
    "\n",
    "for name, model in models.items():\n",
    "    print(f\"Training {name}...\")\n",
    "    model.fit(X_train, y_train)\n",
    "    preds = model.predict(X_test)\n",
    "    rmse = np.sqrt(mean_squared_error(y_test, preds))\n",
    "    results[name] = rmse\n",
    "    print(f\"{name} RMSE: {rmse:.2f}\\n\")\n",
    "\n",
    "# Visualize the model comparisons\n",
    "plt.figure(figsize=(8, 5))\n",
    "sns.barplot(x=list(results.keys()), y=list(results.values()))\n",
    "plt.ylabel('RMSE')\n",
    "plt.title('Model Comparison')\n",
    "plt.xticks(rotation=45)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a816ed1d",
   "metadata": {},
   "source": [
    "## Hyperparameter Tuning Example\n",
    "\n",
    "Below is an example of hyperparameter tuning using GridSearchCV on the Random Forest model. You can adapt and use a similar process for the other models if desired."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "47c47f59",
   "metadata": {},
   "outputs": [],
   "source": [
    "param_grid = {\n",
    "    'n_estimators': [50, 100, 200],\n",
    "    'max_depth': [None, 5, 10, 20]\n",
    "}\n",
    "\n",
    "rf = RandomForestRegressor(random_state=42)\n",
    "grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)\n",
    "grid_search.fit(X_train, y_train)\n",
    "\n",
    "print('Best Parameters:', grid_search.best_params_)\n",
    "best_rf = grid_search.best_estimator_\n",
    "\n",
    "# Evaluate the tuned model\n",
    "tuned_preds = best_rf.predict(X_test)\n",
    "tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))\n",
    "print(f\"Tuned Random Forest RMSE: {tuned_rmse:.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6e8f49bf",
   "metadata": {},
   "source": [
    "## Summary\n",
    "\n",
    "In this notebook, we have:\n",
    "\n",
    "- Loaded and inspected the dataset to verify available columns\n",
    "- Performed exploratory analysis to understand data distributions and detect issues\n",
    "- Generated customer-level features based on a historical period defined by a cutoff date\n",
    "  - Created features such as recency, frequency, monetary value, and tenure\n",
    "  - Computed a target variable (`base_clv`) based on future revenue\n",
    "- Built several regression models to predict CLV and compared their performance\n",
    "- Demonstrated hyperparameter tuning using GridSearchCV\n",
    "\n",
    "Feel free to modify any section based on your dataset structure or business logic."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


{'cells': [{'cell_type': 'markdown',
   'id': '3581629a',
   'metadata': {},
   'source': ['# Customer Lifetime Value (CLV) Prediction Notebook\n',
    '\n',
    'This notebook demonstrates how to build a CLV prediction model using supervised regression models. Unlike an approach that assumes column names, this notebook starts by analyzing the dataset columns and then continues with the following workflow:\n',
    '\n',
    '1. **Data Loading & Analysis:** Read the dataset, display column names, data types, and a preview of the data.\n',
    '2. **Exploratory Data Analysis (EDA):** Generate visualizations and summary statistics to understand the data.\n',
    '3. **Feature Engineering:** Create historical features such as recency, frequency, monetary value, and tenure based on a selected cutoff date. The CLV target (`base_clv`) is calculated using future revenue beyond this cutoff.\n',
    '4. **Model Training & Evaluation:** Train several regression models (Linear Regression, Decision